# Chapter 2 — Build a transformer

A transformer learns from numerical sequences, not raw sentences. This notebook starts by turning the aligned German–English captions in Multi30k into token sequences and integer IDs. By the end, both languages have vocabularies, reversible token-to-ID mappings, and aligned examples ordered by source length for efficient batching.

## Learning goals

You will:

1. download and inspect a parallel corpus;
2. tokenize German and English with language-specific rules;
3. add sequence-boundary tokens and build frequency-based vocabularies;
4. encode tokens as integer IDs and decode them for a sanity check; and
5. preserve source–target alignment while sorting examples by length.

## Shape notation

Preprocessing uses variable-length Python lists, so their *logical* shapes are ragged rather than rectangular:

- one tokenized sentence: `(sequence_length,)`;
- the corpus before padding: `(num_sentences, variable_sequence_length)`;
- a future padded source tensor: `(batch_size, source_sequence_length)`;
- a future padded target tensor: `(batch_size, target_sequence_length)`; and
- embedded transformer input: `(batch_size, sequence_length, embedding_dim)`.

German and English translations do not need to have the same sequence length.


## 1. Download the parallel corpus

Multi30k contains captions paired across languages. The training archive used here provides one German file and one English file. The download is conditional, so rerunning the notebook reuses the local archive; extraction recreates the language files under `files/`.

> The first run requires network access. The extracted dataset is local notebook data and should not be committed unless the project intentionally requires it.


In [1]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NameError: name 'torch' is not defined

In [ ]:
import tarfile
from pathlib import Path

import requests

DATA_DIRECTORY: Path = Path("files")
TRAINING_ARCHIVE: Path = DATA_DIRECTORY / "training.tar.gz"
TRAINING_DATA_URL: str = (
    "https://raw.githubusercontent.com/neychev/"
    "small_DL_repo/master/datasets/Multi30k/training.tar.gz"
)

DATA_DIRECTORY.mkdir(exist_ok=True)

# Reuse the downloaded archive when the notebook is run again.
if not TRAINING_ARCHIVE.exists():
    response: requests.Response = requests.get(TRAINING_DATA_URL)
    with TRAINING_ARCHIVE.open("wb") as archive_file:
        archive_file.write(response.content)

# Extract the paired train.de and train.en files, then close the archive.
with tarfile.open(TRAINING_ARCHIVE) as training_archive:
    training_archive.extractall(DATA_DIRECTORY)

## 2. Load aligned sentence pairs

Each line is one caption. Line order carries the translation relationship: `german_sentences[i]` and `english_sentences[i]` describe the same image. Both files must remain in the same order whenever examples are filtered, shuffled, or sorted.

After UTF-8 decoding and removal of line endings:

- `german_sentences`: `(num_sentences,)` German strings;
- `english_sentences`: `(num_sentences,)` English strings.

In [ ]:
# Matching line numbers identify a German sentence and its English translation.
with (DATA_DIRECTORY / "train.de").open(encoding="utf-8") as german_file:
    german_sentences: list[str] = [line.strip() for line in german_file]
with (DATA_DIRECTORY / "train.en").open(encoding="utf-8") as english_file:
    english_sentences: list[str] = [line.strip() for line in english_file]

### Inspect the corpus before preprocessing

A quick inspection checks two assumptions that later code relies on:

1. both files contain the same number of examples; and
2. rows at matching indices appear to be translations.

The first five pairs also expose punctuation and capitalization that the tokenizers must handle.


In [ ]:
from pprint import pprint

# Equal lengths confirm that every source sentence has a target sentence.
print(f"Number of German sentences: {len(german_sentences)}")
print(f"Number of English sentences: {len(english_sentences)}")
print("First five German sentences:")
pprint(german_sentences[:5])
print("First five English sentences:")
pprint(english_sentences[:5])

## 3. Load language-specific tokenizers

A tokenizer splits text into units such as words and punctuation. spaCy supplies language-specific rules, so German and English use different pipelines. If a model is missing, the cell installs it and retries the load.

The first run may therefore require network access. Subsequent runs load the installed models directly.


In [ ]:
import os

import spacy
from spacy.language import Language


def load_spacy_model(model_name: str) -> Language:
    """Load a spaCy language model, downloading it when necessary.

    Args:
        model_name: Name of the spaCy model to load.

    Returns:
        The loaded language pipeline.
    """
    try:
        return spacy.load(model_name)
    except OSError:
        os.system(f"python -m spacy download {model_name}")
        return spacy.load(model_name)


german_tokenizer: Language = load_spacy_model("de_core_news_sm")
english_tokenizer: Language = load_spacy_model("en_core_web_sm")

### Tokenize one aligned pair

Start with one pair to make the transformation visible. Punctuation becomes its own token. The results are one-dimensional token lists:

- `german_sample_tokens`: `(source_sequence_length,)`;
- `english_sample_tokens`: `(target_sequence_length,)`.

Their lengths can differ because translation does not preserve word count.

In [ ]:
# Keep punctuation tokens so they can receive their own vocabulary IDs.
german_sample_tokens: list[str] = [
    token.text for token in german_tokenizer.tokenizer(german_sentences[0])
]
english_sample_tokens: list[str] = [
    token.text for token in english_tokenizer.tokenizer(english_sentences[0])
]
print(german_sample_tokens)
print(english_sample_tokens)

## 4. Build the English vocabulary

The full English corpus is tokenized first. Every sentence is wrapped with `BOS` (*beginning of sequence*) and `EOS` (*end of sequence*) so the future model can recognize sentence boundaries.

`tokenize_sentences` keeps the language-specific tokenization step in one place. `build_vocabulary` then counts each token and keeps up to 50,000 observed tokens. Two additional IDs have fixed meanings:

| Token | ID | Purpose |
|---|---:|---|
| `PAD` | `0` | Fills unused positions so examples can form a rectangular batch. |
| `UNK` | `1` | Replaces a token that is not present in the vocabulary. |

`enumerate(..., start=2)` assigns ordinary vocabulary IDs from `2` onward, leaving `0` and `1` for the reserved IDs.

- `english_token_sequences`: ragged `(num_sentences, variable_target_sequence_length)`;
- `english_token_to_id`: token → integer ID;
- `english_id_to_token`: integer ID → token;
- `english_vocabulary_size`: observed vocabulary size plus two reserved IDs.

In [ ]:
from collections import Counter
from collections.abc import Sequence

BEGINNING_OF_SEQUENCE: str = "BOS"
END_OF_SEQUENCE: str = "EOS"
PADDING_ID: int = 0
UNKNOWN_ID: int = 1
MAX_VOCABULARY_SIZE: int = 50_000


def tokenize_sentences(sentences: Sequence[str], tokenizer: Language) -> list[list[str]]:
    """Tokenize sentences and add sequence-boundary tokens.

    Args:
        sentences: Sentences to tokenize.
        tokenizer: Language-specific spaCy pipeline.

    Returns:
        Token sequences with `BOS` at the start and `EOS` at the end.
    """
    token_sequences: list[list[str]] = []
    for sentence in sentences:
        sentence_tokens: list[str] = [token.text for token in tokenizer.tokenizer(sentence)]
        token_sequences.append([BEGINNING_OF_SEQUENCE, *sentence_tokens, END_OF_SEQUENCE])
    return token_sequences


def build_vocabulary(
    token_sequences: Sequence[Sequence[str]], max_size: int
) -> tuple[list[tuple[str, int]], dict[str, int], dict[int, str]]:
    """Build frequency-ranked forward and inverse token-ID mappings.

    Args:
        token_sequences: Tokenized sentences used to count token frequencies.
        max_size: Maximum number of observed tokens to keep.

    Returns:
        Token frequencies, a token-to-ID mapping, and its inverse mapping.
    """
    token_counts: Counter[str] = Counter()
    for token_sequence in token_sequences:
        token_counts.update(token_sequence)

    token_frequencies: list[tuple[str, int]] = token_counts.most_common(max_size)
    token_to_id: dict[str, int] = {
        token: token_id for token_id, (token, _) in enumerate(token_frequencies, start=2)
    }
    token_to_id["PAD"] = PADDING_ID
    token_to_id["UNK"] = UNKNOWN_ID
    id_to_token: dict[int, str] = {token_id: token for token, token_id in token_to_id.items()}
    return token_frequencies, token_to_id, id_to_token


# Shape: (num_sentences, variable_target_sequence_length)
english_token_sequences: list[list[str]] = tokenize_sentences(english_sentences, english_tokenizer)
english_token_frequencies: list[tuple[str, int]]
english_token_to_id: dict[str, int]
english_id_to_token: dict[int, str]
(
    english_token_frequencies,
    english_token_to_id,
    english_id_to_token,
) = build_vocabulary(english_token_sequences, MAX_VOCABULARY_SIZE)
english_vocabulary_size: int = len(english_token_frequencies) + 2

print(english_token_frequencies[:5])

### Encode one English sentence

Vocabulary lookup converts the sample tokens into integer IDs. `dict.get(token, UNKNOWN_ID)` supplies ID `1` for a token absent from the capped vocabulary.

`english_sample_ids` has shape `(target_sequence_length,)`. It excludes `BOS` and `EOS` because `english_sample_tokens` came directly from the raw sample; full-corpus sequences in `english_token_sequences` include both markers.

In [ ]:
# Shape: (target_sequence_length,)
english_sample_ids: list[int] = [
    english_token_to_id.get(token, UNKNOWN_ID) for token in english_sample_tokens
]
english_sample_ids

### Decode the English IDs as a sanity check

The inverse dictionary reconstructs tokens from IDs. A successful round trip checks that both mappings agree. The punctuation cleanup improves readability, but it is only a display heuristic—not a general-purpose detokenizer.


In [ ]:
def format_tokens_for_display(tokens: Sequence[str]) -> str:
    """Join tokens and remove spaces before common punctuation.

    This is a display heuristic, not a general-purpose detokenizer.

    Args:
        tokens: Tokens to join into readable text.

    Returns:
        A readable approximation of the original sentence.
    """
    text: str = " ".join(tokens)
    for punctuation in "?:;.,'(\"-!&)%":
        text = text.replace(f" {punctuation}", punctuation)
    return text


# Reverse the lookup to verify that encoding preserved the tokens.
decoded_english_tokens: list[str] = [
    english_id_to_token.get(token_id, "UNK") for token_id in english_sample_ids
]
print(decoded_english_tokens)
print(format_tokens_for_display(decoded_english_tokens))

## 5. Build the German vocabulary

German needs a separate vocabulary because spellings and token frequencies differ by language. Reusing `tokenize_sentences` and `build_vocabulary` makes the parallel processing steps explicit without duplicating their implementation.

- `german_token_sequences`: ragged `(num_sentences, variable_source_sequence_length)`;
- `german_token_to_id`: German token → integer ID;
- `german_id_to_token`: integer ID → German token;
- `german_vocabulary_size`: observed vocabulary size plus two reserved IDs.

In [ ]:
# Shape: (num_sentences, variable_source_sequence_length)
german_token_sequences: list[list[str]] = tokenize_sentences(german_sentences, german_tokenizer)
german_token_frequencies: list[tuple[str, int]]
german_token_to_id: dict[str, int]
german_id_to_token: dict[int, str]
(
    german_token_frequencies,
    german_token_to_id,
    german_id_to_token,
) = build_vocabulary(german_token_sequences, MAX_VOCABULARY_SIZE)
german_vocabulary_size: int = len(german_token_frequencies) + 2

### Check the German mappings

Encode the earlier German sample, then decode it. The ID list has shape `(source_sequence_length,)`. Recovering the original tokens verifies the forward and inverse dictionaries before applying them to the entire corpus.


In [ ]:
# Shape: (source_sequence_length,)
german_sample_ids: list[int] = [
    german_token_to_id.get(token, UNKNOWN_ID) for token in german_sample_tokens
]
german_sample_ids

In [ ]:
# Decode the IDs as a round-trip check of the German mappings.
decoded_german_tokens: list[str] = [
    german_id_to_token.get(token_id, "UNK") for token_id in german_sample_ids
]
print(decoded_german_tokens)
print(format_tokens_for_display(decoded_german_tokens))

## 6. Numericalize and order the full corpus

Every token in every sentence is replaced by its language-specific integer ID:

- `german_id_sequences`: ragged `(num_sentences, variable_source_sequence_length)`;
- `english_id_sequences`: ragged `(num_sentences, variable_target_sequence_length)`.

Sorting by German length groups similarly sized source sequences and can reduce future padding. The same `indices_by_german_length` permutation must be applied to both languages; sorting them independently would destroy translation alignment.

In [ ]:
# Numericalize every sentence; these lists remain ragged until batching.
english_id_sequences: list[list[int]] = [
    [english_token_to_id.get(token, UNKNOWN_ID) for token in token_sequence]
    for token_sequence in english_token_sequences
]
german_id_sequences: list[list[int]] = [
    [german_token_to_id.get(token, UNKNOWN_ID) for token in token_sequence]
    for token_sequence in german_token_sequences
]

# Apply one length-based permutation to both languages to preserve each pair.
indices_by_german_length: list[int] = sorted(
    range(len(german_id_sequences)),
    key=lambda index: len(german_id_sequences[index]),
)
german_id_sequences = [german_id_sequences[index] for index in indices_by_german_length]
english_id_sequences = [english_id_sequences[index] for index in indices_by_german_length]

## 7. Create length-aware batches

The examples are already sorted by German sequence length. Taking contiguous groups therefore places similarly sized source sequences together, which limits padding waste. `batch_start_indices` contains the start offset of each batch and is shuffled so training does not always visit batches from shortest to longest.

- `batch_start_indices`: `(num_batches,)` batch-start offsets;
- each item in `batch_indices`: `(current_batch_size,)` row indices;
- `current_batch_size <= BATCH_SIZE` because the final batch may be smaller.

Only the order of whole batches is randomized. Examples within each batch remain adjacent in the length-sorted corpus.

In [ ]:
import numpy as np
from numpy.typing import NDArray

BATCH_SIZE: int = 128

# Contiguous starts preserve groups of similarly sized German sequences.
batch_start_indices: NDArray[np.int_] = np.arange(0, len(german_id_sequences), BATCH_SIZE)

# Randomize batch order without shuffling translation pairs independently.
np.random.shuffle(batch_start_indices)

batch_indices: list[NDArray[np.int_]] = []
for batch_start_index in batch_start_indices:
    start_index: int = int(batch_start_index)
    stop_index: int = min(len(german_id_sequences), start_index + BATCH_SIZE)
    batch_indices.append(np.arange(start_index, stop_index, dtype=np.int_))

## 8. Pad sequences within a batch

A rectangular array requires every row to have the same length. `pad_sequences` finds the longest sequence in the current batch and appends `PADDING_ID` values to shorter rows.

For a batch containing sequences of different lengths:

- input `sequences`: ragged logical shape `(current_batch_size, variable_sequence_length)`;
- output `padded_sequences`: NumPy shape `(current_batch_size, max_sequence_length_in_batch)`.

Padding only to the longest sequence in each batch uses less memory than padding every example to the longest sequence in the complete corpus. The returned NumPy array can later be converted to a PyTorch integer tensor with the same two-dimensional shape.

In [ ]:
def pad_sequences(
    sequences: Sequence[Sequence[int]], padding_id: int = PADDING_ID
) -> NDArray[np.int_]:
    """Pad integer sequences to the longest sequence in the batch.

    Args:
        sequences: Variable-length integer sequences for one batch.
        padding_id: Token ID appended to shorter sequences.

    Returns:
        A two-dimensional array shaped
        `(current_batch_size, max_sequence_length_in_batch)`.

    Raises:
        ValueError: If `sequences` is empty and therefore has no maximum length.
    """
    max_length: int = max(len(sequence) for sequence in sequences)

    # Shape: (current_batch_size, max_sequence_length_in_batch)
    padded_sequences: NDArray[np.int_] = np.array(
        [list(sequence) + [padding_id] * (max_length - len(sequence)) for sequence in sequences]
    )
    return padded_sequences

## Summary

The preprocessing pipeline now produces batch-ready numerical data for both languages:

- language-specific tokenizers split captions into tokens;
- `BOS` and `EOS` mark sentence boundaries;
- `PADDING_ID = 0` and `UNKNOWN_ID = 1` occupy stable reserved IDs;
- forward and inverse dictionaries support encoding and inspection;
- one shared permutation sorts both languages without breaking translation pairs;
- contiguous index groups form length-aware batches; and
- batch-local padding converts ragged sequences into rectangular NumPy arrays.

A padded source batch has shape `(batch_size, source_sequence_length)` and a padded target batch has shape `(batch_size, target_sequence_length)`, where the final batch may use a smaller first dimension. After conversion to PyTorch tensors, embedding layers will map token IDs to `(batch_size, sequence_length, embedding_dim)` before the sequences enter the transformer.